In [ ]:
import os
import re
import json
import hashlib
import urllib.request
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
with open(os.path.join(BASE_DIR, "config.json"), "r") as f:
    config = json.load(f)

DATA_DIR = config["paths"]["data_dir"]
RESULTS_DIR = config["paths"]["results_dir"]
SPLIT_NAMES = ("train", "dev", "test")
print(f"Loaded configuration. Data directory: {DATA_DIR}")

In [ ]:
URLS = {
    s: f"https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/{s}_sent_emo.csv"
    for s in SPLIT_NAMES
}

data_frames = {}
for split_name, url in URLS.items():
    local_path = os.path.join(DATA_DIR, f"{split_name}_sent_emo.csv")
    if not os.path.exists(local_path) or os.path.getsize(local_path) == 0:
        print(f"Downloading {split_name} split from {url}...")
        tmp_path = local_path + ".part"
        urllib.request.urlretrieve(url, tmp_path)
        os.replace(tmp_path, local_path)
    data_frames[split_name] = pd.read_csv(local_path)
    print(f"Loaded {split_name} split with shape: {data_frames[split_name].shape}")

In [ ]:
REQUIRED_COLS = ["Utterance", "Speaker", "Emotion", "Sentiment", "Dialogue_ID", "Utterance_ID",
                 "Season", "Episode", "StartTime", "EndTime"]
target_emotions = set(config["emotions"])

for split_name, df in data_frames.items():
    missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
    assert not missing_cols, f"[{split_name}] missing columns: {missing_cols}"

    found_emotions = set(df["Emotion"].unique())
    assert found_emotions == target_emotions, (
        f"[{split_name}] emotion labels differ from the 7 target classes. "
        f"Unexpected: {found_emotions - target_emotions}; absent: {target_emotions - found_emotions}"
    )

    exp = config["expected_counts"][split_name]
    n_utt, n_dia = len(df), df["Dialogue_ID"].nunique()
    assert n_utt == exp["utterances"], f"[{split_name}] utterances: {n_utt} != official {exp['utterances']}"
    assert n_dia == exp["dialogues"], f"[{split_name}] dialogues: {n_dia} != official {exp['dialogues']}"
    assert df["Utterance"].notna().all() and df["Speaker"].notna().all(), f"[{split_name}] null utterance/speaker"
    print(f"[{split_name}] OK: {n_utt} utterances, {n_dia} dialogues, 7 emotion classes")

total_utt = sum(len(d) for d in data_frames.values())
total_dia = sum(d["Dialogue_ID"].nunique() for d in data_frames.values())
print(f"Total: {total_utt} utterances, {total_dia} dialogues")

In [ ]:
gap_counts = {}
for split_name, df in data_frames.items():
    assert not df.duplicated(["Dialogue_ID", "Utterance_ID"]).any(), f"[{split_name}] duplicate (Dialogue_ID, Utterance_ID)"

    runs = (df["Dialogue_ID"] != df["Dialogue_ID"].shift()).sum()
    assert runs == df["Dialogue_ID"].nunique(), f"[{split_name}] dialogues are not contiguous in the CSV"

    n_gap = 0
    for did, g in df.groupby("Dialogue_ID", sort=False):
        u = g["Utterance_ID"].tolist()
        assert all(a < b for a, b in zip(u, u[1:])), f"[{split_name}] dialogue {did}: utterances not strictly ascending"
        n_gap += int(u != list(range(len(u))))
    gap_counts[split_name] = n_gap
    assert isinstance(df.index, pd.RangeIndex)
    print(f"[{split_name}] row order verified (contiguous dialogues, ascending utterances; {gap_counts[split_name]} dialogues have Utterance_ID gaps)")

In [ ]:
print("Raw Dialogue_ID ranges (informational only -- these legitimately overlap):")
for s, df in data_frames.items():
    print(f"  {s}: {df['Dialogue_ID'].min()}..{df['Dialogue_ID'].max()}")

def dialogue_fingerprints(df):
    fps = {}
    for did, g in df.groupby("Dialogue_ID", sort=False):
        text = "||".join(f"{sp}::{ut}" for sp, ut in zip(g["Speaker"], g["Utterance"].astype(str)))
        fps[did] = hashlib.sha1(text.encode("utf-8")).hexdigest()
    return fps

def utterance_key(df):
    """Strict utterance identity: clip time + speaker + punctuation/case-insensitive text."""
    norm = df["Utterance"].astype(str).str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
    return list(zip(df["Season"], df["Episode"], df["StartTime"], df["EndTime"], df["Speaker"].str.lower(), norm))

fingerprints = {s: set(dialogue_fingerprints(df).values()) for s, df in data_frames.items()}
pairs = [("train", "dev"), ("train", "test"), ("dev", "test")]

for a, b in pairs:
    shared_dialogues = fingerprints[a] & fingerprints[b]
    assert not shared_dialogues, f"Leakage: {len(shared_dialogues)} identical dialogues shared by {a} and {b}"
    print(f"{a} vs {b}: 0 identical dialogues")

leakage_report = {"identical_dialogues": 0, "utterance_overlaps": []}
keyed = {}
for s_name, df in data_frames.items():
    index = {}
    for k, did, uid in zip(utterance_key(df), df["Dialogue_ID"], df["Utterance_ID"]):
        index.setdefault(k, []).append((int(did), int(uid)))
    keyed[s_name] = index
for a, b in pairs:
    common = set(keyed[a]) & set(keyed[b])
    for k in sorted(common, key=str):
        leakage_report["utterance_overlaps"].append({
            "splits": [a, b], "season": int(k[0]), "episode": int(k[1]), "start": k[2], "speaker": k[4],
            f"{a}_dialogue_utterance": keyed[a][k], f"{b}_dialogue_utterance": keyed[b][k],
        })
    print(f"{a} vs {b}: {len(common)} strictly duplicated utterances (same clip time + speaker + text)")

n_overlap = len(leakage_report["utterance_overlaps"])
if n_overlap:
    print(f"[WARNING] {n_overlap} utterance(s) appear in more than one split. They are NOT removed here; "
          "decide explicitly (e.g. exclude from test) and document it. Details: results/leakage_report.json")
    for o in leakage_report["utterance_overlaps"]:
        print("   ", o)
with open(os.path.join(RESULTS_DIR, "leakage_report.json"), "w") as f:
    json.dump(leakage_report, f, indent=2, default=str)

episodes = {s: set(zip(df["Season"], df["Episode"])) for s, df in data_frames.items()}
for a, b in pairs:
    print(f"(info) episodes shared by {a} and {b}: {len(episodes[a] & episodes[b])}")
print("Dialogue-level split integrity verified (no identical dialogues across splits).")

In [ ]:
order = config["emotions"]
dist = pd.DataFrame({s: df["Emotion"].value_counts().reindex(order).fillna(0).astype(int)
                     for s, df in data_frames.items()})
dist["total"] = dist.sum(axis=1)
dist_pct = (dist[list(SPLIT_NAMES)] / dist[list(SPLIT_NAMES)].sum() * 100).round(2)
print(dist)
print("\nPercentage per split:")
print(dist_pct)

dist.to_csv(os.path.join(RESULTS_DIR, "class_distribution.csv"))
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(order)); w = 0.27
for i, s in enumerate(SPLIT_NAMES):
    ax.bar(x + (i - 1) * w, dist_pct[s].values, w, label=s)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20)
ax.set_ylabel("% of split"); ax.set_title("MELD emotion distribution by official split")
ax.legend(); ax.grid(axis="y", linestyle="--", alpha=0.6)
fig.savefig(os.path.join(RESULTS_DIR, "class_distribution.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved class distribution table and figure.")

In [ ]:
VIDEO_RE = re.compile(r"^dia(\d+)_utt(\d+)\.mp4$")

def classify_dir(path):
    """Map a directory to a split using whole-word tokens (train / dev / test), nearest component first."""
    for part in reversed([p.lower() for p in path.split(os.sep) if p]):
        tokens = set(re.split(r"[^a-z0-9]+", part))
        hits = [s for s in SPLIT_NAMES if s in tokens]
        if len(hits) == 1:
            return hits[0]
        if len(hits) > 1:
            return None
    return None

def discover_media_dirs(search_roots):
    found = {}
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root, followlinks=True):
            n = sum(1 for f in files if VIDEO_RE.match(f))
            if n:
                found[dirpath] = n
    return found

if config.get("media_dirs"):
    media_dirs = dict(config["media_dirs"])
    print("Using media_dirs from config.")
else:
    found = discover_media_dirs(config["media_search_roots"])
    by_split = {s: [] for s in SPLIT_NAMES}
    unclassified = []
    for d, n in found.items():
        s = classify_dir(d)
        (by_split[s] if s else unclassified).append((n, d))
    media_dirs = {}
    for s in SPLIT_NAMES:
        if not by_split[s]:
            raise FileNotFoundError(
                f"No directory with dia*_utt*.mp4 files could be assigned to split '{s}' under "
                f"{config['media_search_roots']}. Directories found: {found}. "
                "Attach the raw MELD videos or set config['media_dirs'] explicitly in Phase 1."
            )
        by_split[s].sort(reverse=True)
        media_dirs[s] = by_split[s][0][1]
        if len(by_split[s]) > 1:
            print(f"[warn] multiple candidate dirs for '{s}', using the largest: {by_split[s]}")
    if unclassified:
        print(f"[info] ignored unclassifiable video directories: {unclassified}")

for s in SPLIT_NAMES:
    print(f"{s}: {media_dirs[s]}")

In [ ]:
MAX_MISSING = config["max_missing_fraction"]
manifest_summary = {}
violations = []

for s in SPLIT_NAMES:
    df = data_frames[s]
    present = {f: os.path.getsize(os.path.join(media_dirs[s], f))
               for f in os.listdir(media_dirs[s]) if VIDEO_RE.match(f)}
    names = "dia" + df["Dialogue_ID"].astype(str) + "_utt" + df["Utterance_ID"].astype(str) + ".mp4"

    status = []
    for n in names:
        if n not in present:
            status.append("missing")
        elif present[n] == 0:
            status.append("empty")
        else:
            status.append("ok")

    manifest = pd.DataFrame({
        "row_idx": np.arange(len(df)),
        "Dialogue_ID": df["Dialogue_ID"].values,
        "Utterance_ID": df["Utterance_ID"].values,
        "video_file": names.values,
        "video_path": [os.path.join(media_dirs[s], n) if st == "ok" else "" for n, st in zip(names, status)],
        "status": status,
    })
    manifest.to_csv(os.path.join(DATA_DIR, f"media_manifest_{s}.csv"), index=False)

    n_bad = int((manifest["status"] != "ok").sum())
    frac = n_bad / len(manifest)
    bad_ids = manifest.loc[manifest["status"] != "ok", ["Dialogue_ID", "Utterance_ID", "status"]].to_dict("records")
    manifest_summary[s] = {"media_dir": media_dirs[s], "n_utterances": len(manifest),
                           "n_unusable": n_bad, "fraction_unusable": frac, "unusable": bad_ids}
    print(f"[{s}] usable clips: {len(manifest) - n_bad}/{len(manifest)}  (unusable: {n_bad}, {frac:.3%})")
    if frac > MAX_MISSING:
        violations.append(f"{s}: {n_bad} unusable clips ({frac:.2%}) > allowed {MAX_MISSING:.2%}")

with open(os.path.join(RESULTS_DIR, "missing_media_report.json"), "w") as f:
    json.dump(manifest_summary, f, indent=2, default=str)

if violations:
    raise RuntimeError(
        "Too many missing/empty video clips -- refusing to continue (no silent zero-vector fallback). "
        + " | ".join(violations)
        + ". Details: results/missing_media_report.json. Check the dataset attachment / media_dirs."
    )
print("Media manifests written; missing-file threshold satisfied.")

In [ ]:
summary = {
    "train_samples": len(data_frames["train"]),
    "dev_samples": len(data_frames["dev"]),
    "test_samples": len(data_frames["test"]),
    "total_utterances": total_utt,
    "total_dialogues": total_dia,
    "leakage_check": "content-based: no identical dialogues across splits",
    "utterance_level_overlaps": n_overlap,
    "row_order_verified": True,
    "dialogues_with_utterance_id_gaps": gap_counts,
    "media_unusable": {s: manifest_summary[s]["n_unusable"] for s in SPLIT_NAMES},
    "status": "verified",
}
with open(os.path.join(DATA_DIR, "dataset_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)
print(json.dumps(summary, indent=2))
print("Phase 2 Dataset Preparation completed successfully.")